In [14]:
# [escape mode]에서: 
# A(above) / B(below) : create new cell
# DD : delete a cell
# M : code mode >> markdown mode
# Y : markdown mode >> code mode

# [enter] : [escape mode] to [edit mode] 

# [edit mode]에서:
# [shift] + [enter] : run current cell & create new cell
# [ctrl] + [enter] : run current cell only

# 3.0 LLMs and Chat models

In [2]:
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI, ChatAnthropic
import os
from dotenv import dotenv_values

env_vars = dotenv_values('.env')

os.environ['OPENAI_API_KEY'] = env_vars.get('OPENAI_API_KEY')

llm = OpenAI()
chat = ChatOpenAI()

# a = llm.predict("How many planets are there?")  ### text-davinci-003 >>> NOT GONNA USE IT! (outdated & expensive)
b = chat.predict("How many planets are there?")  ### gpt-3.5-turbo >>> Newer, Cheaper, and based on davinci-003

# a, b

# 3.1 Predict Messages



* AIMessage : AI가 생성하는 메세지  
* SystemMessage : LLM에 설정을 하기 위한 메세지

In [3]:

from langchain.schema import HumanMessage, AIMessage, SystemMessage



chat = ChatOpenAI(temperature=0.1)

messages = [
    SystemMessage(
        content="You are a geography expert. And you only reply in Italian."
    ),
    AIMessage(content="Ciao, mi chiamo Paolo!"),
    HumanMessage(
        content="What is the distance between Mexico and Thailand? Also what is your name?"
    )
]

chat.predict_messages(messages)

AIMessage(content='La distanza tra il Messico e la Thailandia è di circa 16.000 chilometri. Come posso aiutarti oggi?')

# 3.2 Prompt Templates

* PromptTemplate : string을 이용해 template를 만듦.
* ChatPromptTemplate : template을 message로부터 만듦.

In [3]:
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain.prompts import PromptTemplate, ChatPromptTemplate



chat = ChatOpenAI(temperature=0.1)

template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}?")

prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

'The distance between Mexico and Thailand is approximately 9,500 miles (15,300 kilometers) when measured in a straight line.'

In [4]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. And you only reply in {language}."),
    ("ai", "Ciao, mi chiamo {name}!"),
    ("human", "What is the distance between {country_a} and {country_b}? Also what is your name?")
])

prompt2 = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt2)

AIMessage(content='Γεια σας! Το όνομά μου είναι Σωκράτης. Η απόσταση μεταξύ του Μεξικού και της Ταϊλάνδης είναι περίπου 16.000 χιλιόμετρα.')

# 3.3 OutputParser and LCEL

In [5]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):

    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))


p = CommaOutputParser()

p.parse("Hello, how, are, you")

['Hello', 'how', 'are', 'you']

In [43]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be \
     answered with a comma-seperated list of max {max_items} in lowercase. DO NOT reply with anything else."),
     ("human", "{question}")
])

prompt = template.format_messages(
    max_items=10, question="What are the colors?"
)

result = chat.predict_messages(prompt)

p = CommaOutputParser()

p.parse(result.content)

['red',
 'orange',
 'yellow',
 'green',
 'blue',
 'indigo',
 'violet',
 'black',
 'white',
 'gray']

In [44]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items":5,
    "question": "What are the poketmons?"
})

['pikachu', 'charizard', 'bulbasaur', 'squirtle', 'jigglypuff']

# 3.4 Chaining Chains

### Components of Chain
1. Prompt
2. Retriever
3. LLM, ChatModel
4. Tool
5. OutputParser

---

#### Input Type
* Prompt > Dict
* Retriever > String
* LLM, ChatModel > String, list of chat messages or a PromptValue
* Tool > String, Dict, depending on the tool
* OutputParser > the output of an LLM or ChatModel

#### Output Type
* Prompt > PromptValue
* Retriever > List of documents
* LLM > String
* ChatModel > ChatMessage
* Tool > depending on the tool
* OutputParser > depending on the parser

In [15]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=0.1,
                  streaming=True,
                  callbacks=[StreamingStdOutCallbackHandler()])

chef_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a world-class international chef. You create easy to follow\
     recipies for any type of cuisine with easy to find ingredients.'),
    ('human', 'I want to cook {cuisine} food.')
])

chef_chain = chef_prompt | chat

veg_chef_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a vegetarian chef specialized on making traditional recipies\
     vegetarian. You find alternative ingredients and explain their preparation. You\
     don\'t radically modify the recipe. If there is no alternative for a food just\
     say you don\'t know how to replace it.'),
     ('human', '{recipe}')
])

veg_chain = veg_chef_prompt | chat

final_chain = {'recipe': chef_chain} | veg_chain

final_chain.invoke({'cuisine': 'indian'})

Great choice! Indian cuisine is known for its bold flavors and aromatic spices. Let's start with a classic Indian dish called Butter Chicken. Here's an easy recipe for you:

Ingredients:
- 500g boneless chicken, cut into bite-sized pieces
- 2 tablespoons butter
- 1 onion, finely chopped
- 2 cloves of garlic, minced
- 1-inch piece of ginger, grated
- 2 teaspoons garam masala
- 1 teaspoon turmeric powder
- 1 teaspoon chili powder (adjust according to your spice preference)
- 1 cup tomato puree
- 1/2 cup heavy cream
- Salt, to taste
- Fresh cilantro, for garnish

Instructions:
1. Heat the butter in a large pan over medium heat. Add the chopped onion and sauté until golden brown.
2. Add the minced garlic and grated ginger to the pan. Cook for another minute until fragrant.
3. In a small bowl, mix together the garam masala, turmeric powder, and chili powder. Add this spice mixture to the pan and cook for a minute to toast the spices.
4. Add the chicken pieces to the pan and cook until they 

AIMessageChunk(content="Great choice! Butter Chicken is a delicious and popular Indian dish. To make it vegetarian, we can replace the chicken with a plant-based alternative. Here's an alternative recipe for Vegetarian Butter Chicken:\n\nIngredients:\n- 500g plant-based chicken substitute (such as tofu, tempeh, or seitan), cut into bite-sized pieces\n- 2 tablespoons butter or vegan butter substitute\n- 1 onion, finely chopped\n- 2 cloves of garlic, minced\n- 1-inch piece of ginger, grated\n- 2 teaspoons garam masala\n- 1 teaspoon turmeric powder\n- 1 teaspoon chili powder (adjust according to your spice preference)\n- 1 cup tomato puree\n- 1/2 cup coconut cream or cashew cream (for a creamy texture)\n- Salt, to taste\n- Fresh cilantro, for garnish\n\nInstructions:\n1. Heat the butter in a large pan over medium heat. Add the chopped onion and sauté until golden brown.\n2. Add the minced garlic and grated ginger to the pan. Cook for another minute until fragrant.\n3. In a small bowl, mix

# 4.0 Model I/O

### Modules


<span style="background-color:yellow; color:black;"><strong> Model I/O </strong></span>

Interface with language models

* <strong>Prompts</strong>: Templatize, dynamically select, and manage model inputs
* <strong>Language models</strong>: Make calls to language models through common interfaces
* <strong>Output parsers</strong>: Extract information from model outputs


<span style="background-color:yellow; color:black;"><strong> Retrieval </strong></span>

Interface with application-specific data

How to work with external data, how to provide the data

* <strong>Document loaders</strong>
* <strong>Document transformers</strong>
* <strong>Text embedding models</strong>
* <strong>Vector stores</strong>
* <strong>Retrievers</strong>


<span style="background-color:yellow; color:black;"><strong> Chains </strong></span>

Construct sequences of calls


<span style="background-color:yellow; color:black;"><strong> Agents </strong></span>

Let chains choose which tools to use given high-level directives

The most experimental part.


<span style="background-color:yellow; color:black;"><strong> Memory </strong></span>

Persist application state between runs of a chain


<span style="background-color:yellow; color:black;"><strong> Callbacks </strong></span>

Log and stream intermediate steps of any chain

# 5.5 Memory on LLM Chain

### LLM에 memory를 붙여서 쓰는 방법 

In [5]:

from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
)

template = """
    You are a helpful AI talking to a human.

    {chat_history}
    Human:{question}
    You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True,
)

chain.predict(question="My name is Nico")

chain.predict(question="I live in Seoul")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    
    Human:My name is Nico
    You:


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! How can I assist you today?
    Human:I live in Seoul
    You:


> Finished chain.


"That's great to know! How can I assist you with information or tasks related to Seoul?"

In [6]:
chain.predict(question="What is my name?")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! How can I assist you today?
Human: I live in Seoul
AI: That's great to know! How can I assist you with information or tasks related to Seoul?
    Human:What is my name?
    You:



Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised ServiceUnavailableError: The server is overloaded or not ready yet..



> Finished chain.


'Your name is Nico.'

# 5.6 Chat Based Memory

### 

In [3]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

chain.predict(question="My name is Nico")



> Entering new LLMChain chain...
Prompt after formatting:
System: You are a helpful AI talking to a human
Human: My name is Nico

> Finished chain.


'Nice to meet you, Nico! How can I assist you today?'

# 5.7 LCEL Based Memory

In [7]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)


def load_memory(_):
    return memory.load_memory_variables({})["history"]


chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm


def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result)

In [8]:

invoke_chain("My name is nico")

invoke_chain("What is my name?")

content='Nice to meet you, Nico! How can I assist you today?'
content='Your name is Nico.'


### 6.0 RAG (Retrieval Augumented Generation)

private document를 question과 함께 context 형태로 전달해 해당 document에 대한 내용 추가 학습.
-> tuning과 비슷한 개념?


### Retrieve
source >> load >> transform(split data) >> embed >> store >> retrieve(search) 



# 6.1 Data Loaders and Splitters

In [17]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators='\n',
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
)

# loader = TextLoader("./files/free_text.txt")
loader = UnstructuredFileLoader("./files/chapter_one.pdf")

len(loader.load_and_split(text_splitter=splitter))

16

# 6.2 Tiktoken

OpenAI에 의해 만들어진 tokenizer의 일종.

In [19]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)


loader = UnstructuredFileLoader("./files/chapter_one.docx")

# 6.3 Vectors
<pre>
   (e.g.) <3-dimension vectors>
   : 실제 모델은 1000개 이상의 차원으로 이루어진 벡터 사용.
     수치화에 따른 단어 사이 관계성 추정.

            M     |    F    |    R    
   King    0.9    |   0.1   |   1.0
   Man     0.9    |   0.1   |    0
   ==================================
   (King - Man)

   <span style="background-color:blue;">Royal</span>    0     |    0    |    1.0
   </pre>

In [24]:
from langchain.embeddings import OpenAIEmbeddings


embedder = OpenAIEmbeddings()
embedder.embed_query("Hi")

vector = embedder.embed_documents([
    "Hi",
    "How are you?",
    "longer sentences is allowed because the subject of embedding is a document!"
])

print(len(vector), len(vector[0]))

3 1536


In [25]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import Chroma
from langchain.storage import LocalFileStore

cache_dir = LocalFileStore("./.cache/")


splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/chapter_one.pdf")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

In [22]:
results = vectorstore.similarity_search("where does winston live")

results

Number of requested results 4 is greater than number of elements in index 3, updating n_results = 3


[Document(page_content="It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions (…).\n5\n10\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine (…), went slowly, resting several times on the way. On each landing, the poster with the enormous face gazed from the wall. It

# 6.6 RetrievalQA

### [Legacy] LLMChain
> Recommend using LCEL(LangChain Expression Language)

* Retriever : Interface which makes document retrieve. 

* Chain Types
- Stuff documents chain : 모든 docs를 하나의 context로 만들어 전달.
- Refine documents chain : 여러 개의 docs를 반복하면서 질문에 대한 답을 업데이트하는 방식. 비쌈.
- Map reduce documents chain : 여러 개의 docs에 대해 각각 요약본을 만들어 전달.
- Map re-rank documents chain : 여러 개의 docs에 대해 각각 답변을 생성하고 점수를 매겨 best answer를 반환.

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA
from dotenv import dotenv_values
import os

env_vars = dotenv_values('.env')
os.environ['OPENAI_API_KEY'] = env_vars.get('OPENAI_API_KEY')

llm = ChatOpenAI()

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/free_text.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
)

# chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="map_rerank",
#     retriever=vectorstore.as_retriever(),
# )

chain.run("Describe Victory Mansions")

c:\Users\hihye\anaconda3\lib\site-packages\langchain\chains\llm.py:349: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


"I don't know"

# 6.8 Stuff LCEL Chain

In [8]:

from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/free_text.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriver = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}",
        ),
        ("human", "{question}"),
    ]
)

chain = (
    {
        "context": retriver,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

chain.invoke("Describe Victory Mansions")

AIMessage(content="I don't have information about Victory Mansions in the provided text.")